In [1]:
import pandas as pd
import numpy as np
import json, time, hashlib, os, warnings
warnings.filterwarnings('ignore')

from web3      import Web3
from solcx     import compile_source, install_solc

RES      = '../results/'
MODELS   = '../models/'
CONTRACT = '../contracts/FraudDetection.sol'

# ── Connect to Ganache ─────────────────────────────────────────
GANACHE_URL = 'http://127.0.0.1:7545'
w3          = Web3(Web3.HTTPProvider(GANACHE_URL))

print("=" * 55)
print("GANACHE CONNECTION CHECK")
print("=" * 55)
print(f"Connected:        {w3.is_connected()}")

if not w3.is_connected():
    print()
    print("ERROR: Cannot connect to Ganache.")
    print("Make sure Ganache Desktop is open and running.")
    print("Click 'Quickstart Ethereum' in Ganache.")
    raise ConnectionError("Ganache not running at http://127.0.0.1:7545")

print(f"Chain ID:         {w3.eth.chain_id}")
print(f"Block number:     {w3.eth.block_number}")
print(f"Accounts:         {len(w3.eth.accounts)}")
print()
print("Available accounts:")
for i, acc in enumerate(w3.eth.accounts):
    bal = w3.from_wei(w3.eth.get_balance(acc), 'ether')
    print(f"  [{i}] {acc}  —  {bal:.2f} ETH")

# Use account 0 as the deployer/owner
DEPLOYER = w3.eth.accounts[0]
print(f"\nDeployer account: {DEPLOYER}")

GANACHE CONNECTION CHECK
Connected:        True
Chain ID:         1337
Block number:     1752
Accounts:         10

Available accounts:
  [0] 0x6658A7caE72D6b009CB0b769f6d7F809216042D9  —  100.00 ETH
  [1] 0xcFe60A7d869AE60FED61515AF123F39367909AEc  —  100.00 ETH
  [2] 0xEdE16d85547A400119435f6bBF082563eEF36FE5  —  100.00 ETH
  [3] 0x66F7D601fCDEF2303f191792F93dA800c11091aB  —  100.00 ETH
  [4] 0x02cd947ba927eE3598d79E8fbcF87e730a030a32  —  100.00 ETH
  [5] 0x4De2e463E5403565520bedd77975b13BF3bceeA1  —  100.00 ETH
  [6] 0x4dCD01dA9f4a63a390de605BdC23ecE17d28D347  —  100.00 ETH
  [7] 0x0Ce338512F74325D81A7f055780fc566d19a139b  —  100.00 ETH
  [8] 0x70FFF12305aDD67886219b0E4ca9F635442e2e6b  —  100.00 ETH
  [9] 0x61aBea61654dc937874a8E6eEbF8C50Fb66D3C9c  —  100.00 ETH

Deployer account: 0x6658A7caE72D6b009CB0b769f6d7F809216042D9


In [2]:
print("=" * 55)
print("COMPILING AND DEPLOYING SMART CONTRACT")
print("=" * 55)

# Install Solidity compiler
print("Installing Solidity 0.8.0...")
install_solc('0.8.0')
print("Compiler ready.")

# Read contract source
with open(CONTRACT, 'r') as f:
    source_code = f.read()
print(f"Contract source loaded: {len(source_code)} characters")

# Compile
t_compile = time.time()
compiled  = compile_source(
    source_code,
    output_values=['abi', 'bin'],
    solc_version='0.8.0'
)
compile_time = time.time() - t_compile

contract_id        = '<stdin>:FraudDetection'
contract_interface = compiled[contract_id]
abi                = contract_interface['abi']
bytecode           = contract_interface['bin']

print(f"Compiled in {compile_time:.2f}s")
print(f"ABI entries: {len(abi)}")
print(f"Bytecode length: {len(bytecode)} chars")

# Deploy
print("\nDeploying contract to Ganache...")
t_deploy   = time.time()
Contract   = w3.eth.contract(abi=abi, bytecode=bytecode)
tx_hash    = Contract.constructor().transact({
    'from': DEPLOYER,
    'gas' : 3_000_000
})
receipt      = w3.eth.wait_for_transaction_receipt(tx_hash)
deploy_time  = time.time() - t_deploy

CONTRACT_ADDRESS = receipt['contractAddress']

print(f"\nContract deployed successfully!")
print(f"  Address:      {CONTRACT_ADDRESS}")
print(f"  Gas used:     {receipt['gasUsed']:,}")
print(f"  Block number: {receipt['blockNumber']}")
print(f"  Deploy time:  {deploy_time:.3f}s")

# Save contract info for dashboard
contract_info = {
    'address'  : CONTRACT_ADDRESS,
    'abi'      : abi,
    'deployer' : DEPLOYER,
    'chain_id' : w3.eth.chain_id,
    'deploy_gas': receipt['gasUsed'],
    'deploy_block': receipt['blockNumber'],
}
with open(RES + 'blockchain_config.json', 'w') as f:
    json.dump(contract_info, f, indent=2)
print(f"\nSaved: {RES}blockchain_config.json")

# Instantiate deployed contract
deployed = w3.eth.contract(address=CONTRACT_ADDRESS, abi=abi)
print(f"\nContract instantiated and ready.")
print(f"Total records stored so far: {deployed.functions.getTotalRecords().call()}")

COMPILING AND DEPLOYING SMART CONTRACT
Installing Solidity 0.8.0...
Compiler ready.
Contract source loaded: 2732 characters
Compiled in 0.13s
ABI entries: 10
Bytecode length: 7924 chars

Deploying contract to Ganache...

Contract deployed successfully!
  Address:      0x0f4ff22EfcbE6772155E2aB206d85CA669454F32
  Gas used:     909,053
  Block number: 1753
  Deploy time:  0.135s

Saved: ../results/blockchain_config.json

Contract instantiated and ready.
Total records stored so far: 0


In [3]:
# Redeploy fresh contract
from solcx import compile_source, install_solc
install_solc('0.8.0')
with open(CONTRACT, 'r') as f:
    source_code = f.read()
compiled           = compile_source(source_code, output_values=['abi','bin'], solc_version='0.8.0')
contract_interface = compiled['<stdin>:FraudDetection']
abi                = contract_interface['abi']
bytecode           = contract_interface['bin']

print("Redeploying fresh contract...")
t_deploy  = time.time()
Contract  = w3.eth.contract(abi=abi, bytecode=bytecode)
tx_hash   = Contract.constructor().transact({'from': DEPLOYER, 'gas': 3_000_000})
receipt   = w3.eth.wait_for_transaction_receipt(tx_hash)
deploy_time      = time.time() - t_deploy
CONTRACT_ADDRESS = receipt['contractAddress']
deploy_gas       = receipt['gasUsed']
deployed  = w3.eth.contract(address=CONTRACT_ADDRESS, abi=abi)

contract_info['address']    = CONTRACT_ADDRESS
contract_info['deploy_gas'] = deploy_gas
with open(RES + 'blockchain_config.json', 'w') as f:
    json.dump(contract_info, f, indent=2)

print(f"New contract: {CONTRACT_ADDRESS}")
print(f"Deploy gas:   {deploy_gas:,}")
print(f"Records now:  {deployed.functions.getTotalRecords().call()}")
print()

# ── Store predictions ──────────────────────────────────────────
print("=" * 55)
print("STORING FRAUD PREDICTIONS ON BLOCKCHAIN")
print("=" * 55)

preds_df = pd.read_csv(RES + 'test_predictions.csv')
print(f"Total test predictions: {len(preds_df)}")
print(f"Flagged as fraud:       {preds_df['FraudPrediction'].sum()}")
print()

fraud_df    = preds_df[preds_df['FraudPrediction'] == 1].copy()
# Only store fraud predictions — cleaner audit trail, every block = confirmed fraud
to_store    = fraud_df.copy()
print(f"Records to store: {len(to_store)} (fraud only — all {len(fraud_df)} fraud providers)")
print()

storage_times  = []
gas_per_tx     = []
block_numbers  = []
stored_records = []

for idx, row in to_store.iterrows():
    provider_id  = str(row['Provider'])
    is_fraud     = bool(row['FraudPrediction'])
    probability  = float(row['FraudProbability'])
    risk_cat     = str(row['RiskCategory'])

    # ── FIXED: round before converting to int ──────────────────
    prob_rounded = round(probability, 4)
    prob_int     = int(round(prob_rounded * 10000))

    # Hash uses same rounded value
    data_str  = f"{provider_id}{prob_rounded:.4f}{risk_cat}"
    data_hash = w3.keccak(text=data_str)

    t_store = time.time()
    tx = deployed.functions.storeFraudRecord(
        provider_id,
        is_fraud,
        prob_int,
        risk_cat,
        data_hash
    ).transact({'from': DEPLOYER, 'gas': 300_000})
    receipt_store = w3.eth.wait_for_transaction_receipt(tx)
    store_time    = time.time() - t_store

    storage_times.append(store_time)
    gas_per_tx.append(receipt_store['gasUsed'])
    block_numbers.append(receipt_store['blockNumber'])

    stored_records.append({
        'Provider'        : provider_id,
        'IsFraud'         : is_fraud,
        'FraudProbability': prob_rounded,
        'RiskCategory'    : risk_cat,
        'DataHash'        : data_hash.hex(),
        'BlockNumber'     : receipt_store['blockNumber'],
        'GasUsed'         : receipt_store['gasUsed'],
        'StoreTime'       : round(store_time, 4),
        'TxHash'          : receipt_store['transactionHash'].hex(),
    })

    status = "FRAUD" if is_fraud else "clean"
    print(f"  [{idx+1:>3}] {provider_id:<12} {status:<6} "
          f"prob={prob_rounded:.4f}  gas={receipt_store['gasUsed']:,}  "
          f"time={store_time:.3f}s  block={receipt_store['blockNumber']}")

print()
print(f"Total records stored: {deployed.functions.getTotalRecords().call()}")
print(f"Avg gas per tx:       {np.mean(gas_per_tx):,.0f}")
print(f"Avg store time:       {np.mean(storage_times):.3f}s")

stored_df = pd.DataFrame(stored_records)
stored_df.to_csv(RES + 'blockchain_stored_records.csv', index=False)
print(f"Saved: {RES}blockchain_stored_records.csv")

Redeploying fresh contract...
New contract: 0x0B03AD12c19915993E7b513F3E45765eB196BD8D
Deploy gas:   909,053
Records now:  0

STORING FRAUD PREDICTIONS ON BLOCKCHAIN
Total test predictions: 1353
Flagged as fraud:       134

Records to store: 134 (fraud only — all 134 fraud providers)

  [ 15] PRV51069     FRAUD  prob=0.9980  gas=188,095  time=0.126s  block=1755
  [ 16] PRV51073     FRAUD  prob=0.9999  gas=171,007  time=0.083s  block=1756
  [ 83] PRV51384     FRAUD  prob=0.9886  gas=170,995  time=0.084s  block=1757
  [ 87] PRV51407     FRAUD  prob=0.9645  gas=171,007  time=0.081s  block=1758
  [ 94] PRV51452     FRAUD  prob=0.8859  gas=171,007  time=0.064s  block=1759
  [ 97] PRV51463     FRAUD  prob=0.8355  gas=170,995  time=0.054s  block=1760
  [100] PRV51485     FRAUD  prob=0.9341  gas=171,007  time=0.063s  block=1761
  [154] PRV51782     FRAUD  prob=0.9196  gas=171,007  time=0.097s  block=1762
  [171] PRV51854     FRAUD  prob=0.9908  gas=171,007  time=0.116s  block=1763
  [183] PRV5

In [4]:
print("=" * 55)
print("BLOCKCHAIN RETRIEVAL AND INTEGRITY VERIFICATION")
print("=" * 55)

total = deployed.functions.getTotalRecords().call()
print(f"Total records on chain: {total}")
print()

verified        = 0
failed          = 0
retrieval_times = []
verified_records = []

for record_id in range(total):
    t_retrieve = time.time()

    # Retrieve from blockchain
    (prov, is_fraud, prob_int, risk,
     timestamp, stored_hash) = deployed.functions.getRecord(record_id).call()

    retrieve_time = time.time() - t_retrieve
    retrieval_times.append(retrieve_time)

    # ── FIXED: use integer prob_int directly to avoid float rounding ──
    probability_rounded = round(prob_int / 10000, 4)
    expected_str        = f"{prov}{probability_rounded:.4f}{risk}"
    expected_hash       = w3.keccak(text=expected_str)

    # Verify integrity
    is_valid = (stored_hash == expected_hash)
    if is_valid:
        verified += 1
    else:
        failed += 1

    verified_records.append({
        'RecordID'        : record_id,
        'Provider'        : prov,
        'IsFraud'         : is_fraud,
        'FraudProbability': probability_rounded,
        'RiskCategory'    : risk,
        'Timestamp'       : timestamp,
        'HashMatch'       : is_valid,
        'RetrieveTime'    : round(retrieve_time, 4),
    })

    status = "✓ VALID" if is_valid else "✗ MISMATCH"
    print(f"  Record {record_id:>3}: {prov:<12} fraud={is_fraud}  "
          f"prob={probability_rounded:.4f}  {status}  {retrieve_time:.4f}s")

print()
print("=" * 55)
print("VERIFICATION SUMMARY — FOR PAPER")
print("=" * 55)
print(f"Total records retrieved : {total}")
print(f"Hash verified (valid)   : {verified}  ({verified/total*100:.1f}%)")
print(f"Hash failed (mismatch)  : {failed}")
print(f"Avg retrieval time      : {np.mean(retrieval_times):.4f}s")

if failed == 0:
    print()
    print("INTEGRITY RESULT: 100% records verified.")
    print("Blockchain tamper-proof storage confirmed.")
else:
    print()
    print(f"WARNING: {failed} record(s) could not be verified.")
    print("Check those specific records manually.")

verified_df = pd.DataFrame(verified_records)
verified_df.to_csv(RES + 'blockchain_verified_records.csv', index=False)
print(f"\nSaved: {RES}blockchain_verified_records.csv")

BLOCKCHAIN RETRIEVAL AND INTEGRITY VERIFICATION
Total records on chain: 134

  Record   0: PRV51069     fraud=True  prob=0.9980  ✓ VALID  0.0310s
  Record   1: PRV51073     fraud=True  prob=0.9999  ✓ VALID  0.0283s
  Record   2: PRV51384     fraud=True  prob=0.9886  ✓ VALID  0.0414s
  Record   3: PRV51407     fraud=True  prob=0.9645  ✓ VALID  0.0294s
  Record   4: PRV51452     fraud=True  prob=0.8859  ✓ VALID  0.0346s
  Record   5: PRV51463     fraud=True  prob=0.8355  ✓ VALID  0.0337s
  Record   6: PRV51485     fraud=True  prob=0.9341  ✓ VALID  0.0301s
  Record   7: PRV51782     fraud=True  prob=0.9196  ✓ VALID  0.0376s
  Record   8: PRV51854     fraud=True  prob=0.9908  ✓ VALID  0.0309s
  Record   9: PRV51939     fraud=True  prob=0.9999  ✓ VALID  0.0253s
  Record  10: PRV51982     fraud=True  prob=0.9945  ✓ VALID  0.0296s
  Record  11: PRV51985     fraud=True  prob=0.9529  ✓ VALID  0.0281s
  Record  12: PRV52006     fraud=True  prob=0.9500  ✓ VALID  0.0331s
  Record  13: PRV52078    

In [5]:
print("=" * 60)
print("BLOCKCHAIN RUNTIME METRICS — ADD TO PAPER TABLE III")
print("=" * 60)

avg_store_time    = np.mean(storage_times)
avg_gas           = np.mean(gas_per_tx)
avg_retrieve_time = np.mean(retrieval_times)
deploy_gas        = contract_info['deploy_gas']

print(f"\nContract deployment:")
print(f"  Gas used:           {deploy_gas:,}")
print(f"  Deploy time:        {deploy_time:.3f}s")
print()
print(f"Per-record storage:")
print(f"  Avg gas per tx:     {avg_gas:,.0f}")
print(f"  Avg store time:     {avg_store_time:.3f}s")
print(f"  Min store time:     {np.min(storage_times):.3f}s")
print(f"  Max store time:     {np.max(storage_times):.3f}s")
print()
print(f"Per-record retrieval:")
print(f"  Avg retrieve time:  {avg_retrieve_time:.4f}s")
print()

# Load existing runtime table and add blockchain rows
runtime_df = pd.read_csv(RES + 'runtime_table.csv')

bc_rows = pd.DataFrame([
    {
        'Component'       : 'Blockchain Contract Deployment',
        'Time (seconds)'  : round(deploy_time, 3),
        'Notes'           : f'Gas used: {deploy_gas:,}'
    },
    {
        'Component'       : 'Blockchain Storage (per record)',
        'Time (seconds)'  : round(avg_store_time, 3),
        'Notes'           : f'Avg gas: {avg_gas:,.0f}'
    },
    {
        'Component'       : 'Blockchain Retrieval (per record)',
        'Time (seconds)'  : round(avg_retrieve_time, 4),
        'Notes'           : 'Read-only call (no gas)'
    },
])

runtime_complete = pd.concat([runtime_df, bc_rows], ignore_index=True)
runtime_complete.to_csv(RES + 'runtime_table_complete.csv', index=False)

print("COMPLETE RUNTIME TABLE (for Paper Table III):")
print(runtime_complete.to_string(index=False))
print(f"\nSaved: {RES}runtime_table_complete.csv")

BLOCKCHAIN RUNTIME METRICS — ADD TO PAPER TABLE III

Contract deployment:
  Gas used:           909,053
  Deploy time:        0.048s

Per-record storage:
  Avg gas per tx:     171,133
  Avg store time:     0.071s
  Min store time:     0.047s
  Max store time:     0.126s

Per-record retrieval:
  Avg retrieve time:  0.0293s

COMPLETE RUNTIME TABLE (for Paper Table III):
                Model  Train Time (s)  Pred Time (s)  Val AUC (%)  Samples Trained On  Samples Predicted                         Component  Time (seconds)                   Notes
        Random Forest           1.523         0.0898        96.84              7846.0             1082.0                               NaN             NaN                     NaN
              XGBoost           0.769         0.0036        96.79              7846.0             1082.0                               NaN             NaN                     NaN
Logistic Regression ★           0.237         0.0000        96.75              7846.0       

In [6]:
print("=" * 60)
print("PHASE 5 COMPLETE — BLOCKCHAIN SUMMARY FOR PAPER")
print("=" * 60)

print(f"""
BLOCKCHAIN TECHNICAL DETAILS (Section — Blockchain Architecture):

  Platform:           Ethereum (local Ganache testnet)
  Smart Contract:     FraudDetection.sol (Solidity ^0.8.0)
  Contract Address:   {CONTRACT_ADDRESS}
  Deployer:           {DEPLOYER}
  Chain ID:           {w3.eth.chain_id}

  Deployment:
    Gas used:         {deploy_gas:,}
    Deploy time:      {deploy_time:.3f}s

  Storage:
    Records stored:   {total}
    Avg gas/record:   {avg_gas:,.0f}
    Avg store time:   {avg_store_time:.3f}s

  Retrieval:
    Records verified: {verified}/{total} ({verified/total*100:.1f}%)
    Avg retrieve:     {avg_retrieve_time:.4f}s
    Tampered records: {failed}

  Smart Contract Functions:
    storeFraudRecord() — stores prediction with hash
    getRecord()        — retrieves any stored record
    verifyRecord()     — on-chain hash verification
    getTotalRecords()  — returns total stored count

  Data stored per record:
    providerID, isFraud, fraudProbability (×10000),
    riskCategory, timestamp, dataHash (keccak256)
""")

print("Results saved to results/:")
for f in sorted(os.listdir(RES)):
    if 'blockchain' in f or 'runtime' in f:
        size = os.path.getsize(RES + f) / 1024
        print(f"  {f:<45} {size:.1f} KB")

print()


PHASE 5 COMPLETE — BLOCKCHAIN SUMMARY FOR PAPER

BLOCKCHAIN TECHNICAL DETAILS (Section — Blockchain Architecture):

  Platform:           Ethereum (local Ganache testnet)
  Smart Contract:     FraudDetection.sol (Solidity ^0.8.0)
  Contract Address:   0x0B03AD12c19915993E7b513F3E45765eB196BD8D
  Deployer:           0x6658A7caE72D6b009CB0b769f6d7F809216042D9
  Chain ID:           1337

  Deployment:
    Gas used:         909,053
    Deploy time:      0.048s

  Storage:
    Records stored:   134
    Avg gas/record:   171,133
    Avg store time:   0.071s

  Retrieval:
    Records verified: 134/134 (100.0%)
    Avg retrieve:     0.0293s
    Tampered records: 0

  Smart Contract Functions:
    storeFraudRecord() — stores prediction with hash
    getRecord()        — retrieves any stored record
    verifyRecord()     — on-chain hash verification
    getTotalRecords()  — returns total stored count

  Data stored per record:
    providerID, isFraud, fraudProbability (×10000),
    riskCategory,

In [7]:
print("=" * 60)
print("MODEL SANITY CHECK — test_predictions.csv")
print("=" * 60)

preds_df = pd.read_csv(RES + 'test_predictions.csv')

print(f"\nTotal test providers:     {len(preds_df)}")
print(f"Predicted fraud:          {preds_df['FraudPrediction'].sum()} ({preds_df['FraudPrediction'].mean()*100:.1f}%)")
print(f"Predicted non-fraud:      {(preds_df['FraudPrediction']==0).sum()} ({(preds_df['FraudPrediction']==0).mean()*100:.1f}%)")

print(f"\nRisk category breakdown:")
print(preds_df['RiskCategory'].value_counts().to_string())

print(f"\nFraud probability stats:")
print(preds_df['FraudProbability'].describe().round(4).to_string())

print(f"\nFraud probability distribution:")
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
preds_df['prob_bin'] = pd.cut(preds_df['FraudProbability'], bins=bins)
print(preds_df['prob_bin'].value_counts().sort_index().to_string())

print()
# Expected: ~9-10% fraud rate on test set (similar to training distribution)
expected_rate = 0.094
actual_rate   = preds_df['FraudPrediction'].mean()
diff          = abs(actual_rate - expected_rate)

print(f"Expected fraud rate: ~{expected_rate*100:.1f}%  (same as training distribution)")
print(f"Actual fraud rate:   {actual_rate*100:.1f}%")
print()
if diff < 0.05:
    print("STATUS: ✓ NORMAL — prediction rate matches training distribution")
    print("Model is working correctly.")
elif actual_rate > 0.20:
    print("STATUS: ⚠ HIGH — model is flagging too many providers")
    print("Check if scaler was applied correctly to test set")
elif actual_rate < 0.03:
    print("STATUS: ⚠ LOW — model is flagging too few providers")
    print("Check if test features were engineered the same way as train")
else:
    print("STATUS: ✓ ACCEPTABLE — slight deviation but within normal range")

MODEL SANITY CHECK — test_predictions.csv

Total test providers:     1353
Predicted fraud:          134 (9.9%)
Predicted non-fraud:      1219 (90.1%)

Risk category breakdown:
RiskCategory
Low       1097
High       172
Medium      84

Fraud probability stats:
count    1353.0000
mean        0.2127
std         0.2972
min         0.0000
25%         0.0393
50%         0.0574
75%         0.2217
max         1.0000

Fraud probability distribution:
prob_bin
(0.0, 0.1]    868
(0.1, 0.2]    128
(0.2, 0.3]     56
(0.3, 0.4]     44
(0.4, 0.5]     25
(0.5, 0.6]     26
(0.6, 0.7]     33
(0.7, 0.8]     34
(0.8, 0.9]     39
(0.9, 1.0]     99

Expected fraud rate: ~9.4%  (same as training distribution)
Actual fraud rate:   9.9%

STATUS: ✓ NORMAL — prediction rate matches training distribution
Model is working correctly.


In [8]:
print(deployed.functions.getTotalRecords().call())

134


In [9]:
print(f"Total records stored on chain: {deployed.functions.getTotalRecords().call()}")

# Verify a few specific records
for record_id in [0, 50, 100, 138]:
    try:
        record = deployed.functions.getRecord(record_id).call()
        print(f"Record {record_id}: {record[0]} | fraud={record[1]} | prob={record[2]/10000:.4f}")
    except Exception as e:
        print(f"Record {record_id}: ERROR — {e}")

Total records stored on chain: 134
Record 0: PRV51069 | fraud=True | prob=0.9980
Record 50: PRV54003 | fraud=True | prob=0.8471
Record 100: PRV55845 | fraud=True | prob=0.9533
Record 138: ERROR — {'message': 'VM Exception while processing transaction: revert Record does not exist', 'stack': 'CallError: VM Exception while processing transaction: revert Record does not exist\n    at Blockchain.simulateTransaction (C:\\Program Files\\WindowsApps\\GanacheUI_2.7.1.0_x64__rb4352f0jd4m2\\app\\resources\\static\\node\\node_modules\\ganache\\dist\\node\\1.js:2:72658)', 'code': -32000, 'name': 'CallError', 'data': '0x08c379a0000000000000000000000000000000000000000000000000000000000000002000000000000000000000000000000000000000000000000000000000000000155265636f726420646f6573206e6f742065786973740000000000000000000000'}
